# STT 08 — Évaluation sur le test officiel (*Scripted Speech 27.0* FR)

**But :** mesurer **une seule fois** le WER et le CER du meilleur modèle (`best.pt`, époque 19, WER dev 37,4 %) sur le **test officiel** : 16 207 clips, des voix jamais entendues, jamais utilisées pour choisir quoi que ce soit.

**Pourquoi le test, et pas le dev ?** Le dev a servi à choisir `best.pt` (le meilleur des 40 points d'évaluation). Ce choix favorise un peu le dev : son WER est légèrement optimiste. Le test n'a jamais servi à rien : c'est le **chiffre honnête** à annoncer.

**Décisions (journal du 12/11) :**
- **Tous les clips gardés par `load_split`**, sans le filtre du CTC : sur le robot, on ne pourra pas trier les phrases à l'avance. On compte quand même les clips « impossibles » pour l'information ;
- **GPU** (environ 30 min de quota au total, dont 22 min de téléchargement).

**Avant de lancer (panneau de droite) :**
1. **Accelerator → GPU T4 x2. Internet → On. Secrets :** coche `MDC_API_KEY`. Notebook en **Private**.
2. **Add Input → Your Work → le notebook d'entraînement** (`notebook3bfc7d7729`, dernière version) : il contient `best.pt`. Si `junior` est aussi en entrée, ce n'est pas grave : la cellule 4 prend le `best.pt` qui a **le plus petit WER dev**.
3. **Save Version → Save & Run All.**

**Règles de données (conditions MDC) :** les données vont dans `/tmp`, jamais sauvegardé avec le notebook. **Aucune phrase du dataset n'est affichée** : seulement des chiffres, aussi écrits dans `test_result.json` (*Output*).

## Cellule 1 — Notre code et la machine

Comme dans `stt_07`.

In [ ]:
import time
SESSION_START = time.time()

!git clone -b stt-transformer-junior --depth 1 https://github.com/mohamed-943/projet-J-ONE.git
%cd projet-J-ONE/recherche/stt_transformer
!pip install -q datacollective soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cellule 2 — Téléchargement et décompression (environ 22 min)

Exactement comme dans `stt_07` : il faut l'archive entière (30,65 Go), même si on n'utilise que le test.

In [ ]:
import os, glob, json, subprocess
from kaggle_secrets import UserSecretsClient

# La clé est lue dans le Secret Kaggle : elle n'apparaît jamais dans le code.
secrets = UserSecretsClient()
try:
    os.environ["MDC_API_KEY"] = secrets.get_secret("MDC_API_KEY")
except Exception:
    os.environ["MDC_API_KEY"] = secrets.get_secret("mdc_api_key")
from datacollective import download_dataset

BIG_ID = "cmu5kub4600pkmh078n7n5ogl"           # Scripted Speech 27.0 FR
BIG_DIR = "/tmp/big"                           # /tmp : jamais sauvegardé avec le notebook (conditions MDC)

download_dataset(BIG_ID, download_directory=BIG_DIR)
archive = glob.glob(f"{BIG_DIR}/*.tar.gz")[0]
subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR], check=True)
os.remove(archive)                             # libère les 30 Go de l'archive
BASE = os.path.dirname(glob.glob(f"{BIG_DIR}/**/train.tsv", recursive=True)[0])
print(f"Données prêtes après {(time.time() - SESSION_START) / 60:.1f} min de session")

## Cellule 3 — Le test officiel

- `load_split` lit `test.tsv`, normalise le texte et retire les phrases avec des caractères hors de notre vocabulaire. On affiche **combien** de clips sont retirés : il faudra l'indiquer à côté du WER.
- On récupère la **durée** de chaque clip (`clip_durations.tsv`) pour le WER par tranche de durée.
- On **compte** les clips « impossibles » pour le CTC (plus de 25 instants nécessaires par seconde), **sans les retirer** (décision a).
- `shuffle=False` : l'ordre ne change pas le résultat, mais il rend l'évaluation reproductible.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from stt.dataset import load_split, CommonVoiceDataset, collate_batch

n_tsv = len(pd.read_csv(f"{BASE}/test.tsv", sep="\t", usecols=["path"]))   # clips du test officiel, avant tout filtre
test_table = load_split([BASE], "test")
print(f"Test : {n_tsv:,} clips dans test.tsv -> {len(test_table):,} gardés par load_split "
      f"({n_tsv - len(test_table):,} retirés, soit {100 * (n_tsv - len(test_table)) / n_tsv:.2f} %, caractères hors vocabulaire)")

# Durée de chaque clip, en secondes (le fichier est en millisecondes)
d = pd.read_csv(f"{BASE}/clip_durations.tsv", sep="\t")
durations = dict(zip(d.iloc[:, 0], d.iloc[:, 1] / 1000))
test_table["seconds"] = test_table["audio_path"].map(lambda p: durations.get(os.path.basename(p)))
print(f"Durée inconnue pour {test_table['seconds'].isna().sum()} clips | durée totale : {test_table['seconds'].sum() / 3600:.1f} h")

# Clips « impossibles » pour le CTC : comptés, mais gardés (décision a)
def ctc_needed(text):
    return len(text) + sum(a == b for a, b in zip(text, text[1:]))
rate = test_table["text"].map(ctc_needed) / test_table["seconds"]
print(f"Clips impossibles pour le CTC (gardés quand même) : {(rate > 25).sum()}")

test_loader = DataLoader(CommonVoiceDataset(test_table, augment=False), batch_size=32,
                         shuffle=False, collate_fn=collate_batch, num_workers=4)

## Cellule 4 — Le meilleur modèle

On cherche tous les `best.pt` des entrées et on garde celui qui a **le plus petit WER dev** : c'est celui de l'époque 19 (37,38 %), pas l'ancien de `junior` (61,9 %).

`model.eval()` coupe le dropout : on mesure le vrai modèle. Il n'y a ni optimizer ni scheduler : on ne fait **aucun apprentissage** ici.

In [ ]:
from stt.model import SpeechTransformer

device = "cuda"
found = sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True))
assert found, "Aucun best.pt : ajoute la sortie du notebook d'entraînement avec Add Input"
checkpoints = [(torch.load(p, map_location=device, weights_only=False), p) for p in found]
print("best.pt trouvés :", [(p, c["step"], round(c["wer"], 2)) for c, p in checkpoints])
ckpt, path = min(checkpoints, key=lambda c: c[0]["wer"])     # le plus petit WER dev
del checkpoints

model = SpeechTransformer().to(device)                       # taille S, comme à l'entraînement
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Modèle chargé : {path} | step {ckpt['step']:,} | WER dev {ckpt['wer']:.2f} %")

## Cellule 5 — L'évaluation (environ 2 min sur GPU)

On n'utilise pas `evaluate()` de `stt/training.py`, pour deux raisons :
1. elle renvoie 3 exemples de phrases, qu'on ne veut **pas** afficher ;
2. on veut les erreurs **clip par clip**, pour les regrouper par durée.

Pour chaque clip, on garde seulement des **nombres** : erreurs de mots, nombre de mots de la référence, erreurs de caractères, nombre de caractères, durée. Le WER global reste le calcul standard : **total des erreurs ÷ total des mots** de toutes les références, comme dans `error_rate`.

In [ ]:
from stt.metrics import greedy_decode, edit_distance
from stt.text import decode

ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=True)
rows, total_loss, n_batches, t0 = [], 0.0, 0, time.time()

with torch.no_grad():                                        # pas de gradients : évaluation seulement
    for batch in test_loader:
        features, feature_lengths, targets, target_lengths = [t.to(device) for t in batch]
        log_probs, out_lengths = model(features, feature_lengths)
        total_loss += ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths).item()
        n_batches += 1
        hyps = greedy_decode(log_probs, out_lengths)
        for b, hyp in enumerate(hyps):
            ref = decode(targets[b, :target_lengths[b]].tolist())
            rows.append({"word_err": edit_distance(ref.split(), hyp.split()), "words": len(ref.split()),
                         "char_err": edit_distance(list(ref), list(hyp)), "chars": len(ref)})
            # ref et hyp ne sont ni affichés ni sauvegardés (conditions MDC, repo public)

res = pd.DataFrame(rows)
res["seconds"] = test_table["seconds"].values                # même ordre que test_table (shuffle=False)
print(f"{len(res):,} clips évalués en {(time.time() - t0) / 60:.1f} min")

## Cellule 6 — Les résultats

- **WER et CER globaux** : le chiffre à annoncer.
- **Par tranche de durée** : les clips courts ont peu de mots, donc une seule erreur pèse lourd ; les longs demandent au modèle de tenir sur la durée.
- **Part des clips parfaits** (0 erreur de mot) : une lecture plus intuitive.

Tout est écrit dans `test_result.json` (*Output*) : **uniquement des chiffres**.

In [ ]:
def rates(df):
    return {"clips": len(df),
            "wer": round(100 * df["word_err"].sum() / df["words"].sum(), 2),
            "cer": round(100 * df["char_err"].sum() / df["chars"].sum(), 2)}

overall = rates(res)
overall["loss"] = round(total_loss / n_batches, 4)
overall["perfect_clips_pct"] = round(100 * (res["word_err"] == 0).mean(), 1)
print(f"TEST OFFICIEL | WER {overall['wer']:.2f} % | CER {overall['cer']:.2f} % | loss {overall['loss']:.3f} | "
      f"clips sans aucune erreur de mot : {overall['perfect_clips_pct']:.1f} %")
print(f"(rappel, dev officiel : WER {ckpt['wer']:.2f} %)")

bins = [0, 3, 5, 7, float("inf")]
labels = ["< 3 s", "3-5 s", "5-7 s", ">= 7 s"]
res["tranche"] = pd.cut(res["seconds"], bins=bins, labels=labels, right=False)
by_duration = {}
print("\nPar durée de clip :")
for label in labels:
    part = res[res["tranche"] == label]
    if len(part):
        by_duration[label] = rates(part)
        r = by_duration[label]
        print(f"  {label:>7} | {r['clips']:>6,} clips | WER {r['wer']:6.2f} % | CER {r['cer']:6.2f} %")

result = {"checkpoint": {"path": path, "step": ckpt["step"], "dev_wer": round(ckpt["wer"], 2)},
          "test_clips_in_tsv": n_tsv, "test_clips_kept": len(test_table),
          "ctc_impossible_kept": int((rate > 25).sum()),
          "overall": overall, "by_duration": by_duration}
with open("/kaggle/working/test_result.json", "w") as f:
    json.dump(result, f, indent=1)
print("\ntest_result.json écrit dans Output (aucune phrase, uniquement des chiffres)")